## Preparação do ambiente

In [2]:
!pip install -q pyspark duckdb

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import duckdb

## Obtenção do conjunto de dados


In [ ]:
!mkdir -p dataset/zip/

In [ ]:
!curl -L -o /content/dataset/zip/animal-crossing-new-horizons-nookplaza-dataset.zip https://www.kaggle.com/api/v1/datasets/download/jessicali9530/animal-crossing-new-horizons-nookplaza-dataset

In [ ]:
!unzip /content/dataset/zip/animal-crossing-new-horizons-nookplaza-dataset.zip -d /content/dataset/

## Criando uma spark session

In [4]:
spark = SparkSession.builder \
    .appName("cozy-etl") \
    .master("local[*]") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

## Criando nossa camada bronze

Ingestão dos csvs

In [5]:
acessorios = spark.read.csv("/content/dataset/accessories.csv", header=True, inferSchema=True)
conquistas = spark.read.csv("/content/dataset/achievements.csv", header=True, inferSchema=True)
bolsas = spark.read.csv("/content/dataset/bags.csv", header=True, inferSchema=True)
vestimentas = spark.read.csv("/content/dataset/dress-up.csv", header=True, inferSchema=True)
artes = spark.read.csv("/content/dataset/art.csv", header=True, inferSchema=True)
cabeca = spark.read.csv("/content/dataset/headwear.csv", header=True, inferSchema=True)
calca = spark.read.csv("/content/dataset/bottoms.csv", header=True, inferSchema=True)
musica = spark.read.csv("/content/dataset/music.csv", header=True, inferSchema=True)
personagens = spark.read.csv("/content/dataset/villagers.csv", header=True, inferSchema=True)
ferramentas = spark.read.csv("/content/dataset/tools.csv", header=True, inferSchema=True)
receitas = spark.read.csv("/content/dataset/recipes.csv", header=True, inferSchema=True)

Adição de colunas de controle



In [6]:
acessorios = acessorios.withColumn('dt_ingestao', F.current_timestamp())
conquistas = conquistas.withColumn('dt_ingestao', F.current_timestamp())
bolsas = bolsas.withColumn('dt_ingestao', F.current_timestamp())
vestimentas = vestimentas.withColumn('dt_ingestao', F.current_timestamp())
artes = artes.withColumn('dt_ingestao', F.current_timestamp())
cabeca = cabeca.withColumn('dt_ingestao', F.current_timestamp())
calca = calca.withColumn('dt_ingestao', F.current_timestamp())
musica = musica.withColumn('dt_ingestao', F.current_timestamp())
personagens = personagens.withColumn('dt_ingestao', F.current_timestamp())
ferramentas = ferramentas.withColumn('dt_ingestao', F.current_timestamp())
receitas = receitas.withColumn('dt_ingestao', F.current_timestamp())

In [11]:
acessorios.write.parquet("/content/lakehouse/bronze/acessorios", mode = "overwrite")
conquistas.write.parquet("/content/lakehouse/bronze/conquistas", mode = "overwrite")
bolsas.write.parquet("/content/lakehouse/bronze/bolsas", mode = "overwrite")
vestimentas.write.parquet("/content/lakehouse/bronze/vestimentas", mode = "overwrite")
artes.write.parquet("/content/lakehouse/bronze/artes", mode = "overwrite")
cabeca.write.parquet("/content/lakehouse/bronze/cabeca", mode = "overwrite")
calca.write.parquet("/content/lakehouse/bronze/calca", mode = "overwrite")
musica.write.parquet("/content/lakehouse/bronze/musica", mode = "overwrite")
personagens.write.parquet("/content/lakehouse/bronze/personagens", mode = "overwrite")
ferramentas.write.parquet("/content/lakehouse/bronze/ferramentas", mode = "overwrite")
receitas.write.parquet("/content/lakehouse/bronze/receitas", mode = "overwrite")

Adicionando uma camada de persistência



In [12]:
con = duckdb.connect("cozy-lakehouse.db")

In [13]:
con.execute("""CREATE SCHEMA IF NOT EXISTS bronze""")

In [14]:
con.execute("""CREATE TABLE bronze.acessorios AS
SELECT *
FROM '/content/lakehouse/bronze/acessorios/*.parquet'""")
con.execute("""CREATE TABLE bronze.conquistas AS
SELECT *
FROM '/content/lakehouse/bronze/conquistas/*.parquet'""")
con.execute("""CREATE TABLE bronze.bolsas AS
SELECT *
FROM '/content/lakehouse/bronze/bolsas/*.parquet'""")
con.execute("""CREATE TABLE bronze.vestimentas AS
SELECT *
FROM '/content/lakehouse/bronze/vestimentas/*.parquet'""")
con.execute("""CREATE TABLE bronze.artes AS
SELECT *
FROM '/content/lakehouse/bronze/artes/*.parquet'""")
con.execute("""CREATE TABLE bronze.cabeca AS
SELECT *
FROM '/content/lakehouse/bronze/cabeca/*.parquet'""")
con.execute("""CREATE TABLE bronze.calca AS
SELECT *
FROM '/content/lakehouse/bronze/calca/*.parquet'""")
con.execute("""CREATE TABLE bronze.musica AS
SELECT *
FROM '/content/lakehouse/bronze/musica/*.parquet'""")
con.execute("""CREATE TABLE bronze.personagens AS
SELECT *
FROM '/content/lakehouse/bronze/personagens/*.parquet'""")
con.execute("""CREATE TABLE bronze.ferramentas AS
SELECT *
FROM '/content/lakehouse/bronze/ferramentas/*.parquet'""")
con.execute("""CREATE TABLE bronze.receitas AS
SELECT *
FROM '/content/lakehouse/bronze/receitas/*.parquet'""")

In [15]:
con.sql("SELECT * FROM bronze.acessorios LIMIT 5").show()

┌─────────────────┬───────────┬─────────┬─────────┬───────┬─────────┬──────────┬─────────┬─────────────┬──────────────┬───────────────────────────────────────────────────────────────────┬───────────────────────┬─────────────────┬─────────┬──────────┬───────────────────────────────┬─────────────────────────────┬─────────────────────┬──────────────┬───────────────────────────┬─────────────┬───────────────────┬────────────────────────────┐
│      Name       │ Variation │   DIY   │   Buy   │ Sell  │ Color 1 │ Color 2  │  Size   │ Miles Price │    Source    │                           Source Notes                            │ Seasonal Availability │ Mannequin Piece │ Version │  Style   │         Label Themes          │            Type             │ Villager Equippable │   Catalog    │         Filename          │ Internal ID │  Unique Entry ID  │        dt_ingestao         │
│     varchar     │  varchar  │ varchar │ varchar │ int32 │ varchar │ varchar  │ varchar │   varchar   │   varchar    

In [16]:
con.sql("SELECT * FROM bronze.cabeca LIMIT 5").show()

┌────────────────┬───────────┬─────────┬─────────┬───────┬─────────┬─────────┬─────────┬─────────────┬──────────────┬───────────────────────────────────────────────────────────────────┬───────────────────────┬─────────────────┬─────────┬─────────┬─────────────────┬─────────┬─────────────────────┬──────────┬──────────────┬─────────────┬───────────────────┬────────────────────────────┐
│      Name      │ Variation │   DIY   │   Buy   │ Sell  │ Color 1 │ Color 2 │  Size   │ Miles Price │    Source    │                           Source Notes                            │ Seasonal Availability │ Mannequin Piece │ Version │  Style  │  Label Themes   │  Type   │ Villager Equippable │ Catalog  │   Filename   │ Internal ID │  Unique Entry ID  │        dt_ingestao         │
│    varchar     │  varchar  │ varchar │ varchar │ int32 │ varchar │ varchar │ varchar │   varchar   │   varchar    │                              varchar                              │        varchar        │     varchar     

## Refinando a qualidade dos dados - camada silver

In [17]:
con.execute("""CREATE SCHEMA IF NOT EXISTS silver""")

### Agrupamento de semelhantes

Busca de semelhantes

In [20]:
con.sql("""DESCRIBE bronze.acessorios""").show()

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Name                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Variation             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ DIY                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Buy                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Sell                  │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ Color 1               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Color 2               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Size                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Miles Price           │ VARCHAR     │ 

In [21]:
con.sql("""SELECT DISTINCT "Seasonal Availability" FROM bronze.vestimentas""").show()

┌───────────────────────┐
│ Seasonal Availability │
│        varchar        │
├───────────────────────┤
│ All Year              │
│ Winter                │
│ Summer                │
└───────────────────────┘



Agrupamento das tabelas relacionadas

In [22]:
con.execute(
    """
    CREATE TABLE IF NOT EXISTS silver.vestuario AS
    WITH agrupamento AS (
      SELECT
        "Name" as nm_item,
        "Variation" as nm_variacao,
        CASE "DIY"
          WHEN 'YES'
            THEN 'S'
          WHEN 'NO'
            THEN 'N'
          ELSE 'N'
        END as sn_reprod,
        CASE WHEN "Buy" = 'NFS'
          THEN 0
          ELSE CAST("Buy" as DECIMAL(10,2))
        END AS vl_compra,
        CASE
          WHEN "Color 1" = "Color 2"
            THEN "Color 1"
          ELSE "Color 1" || ', ' || "Color 2"
        END AS nm_cores_disponiveis,
        "Size" as tamanho,
        "Source" as nm_origem,
        CASE "Seasonal Availability"
          WHEN 'All Year' THEN 'Constante'
          WHEN 'Winter' THEN 'Inverno'
          WHEN 'Summer' THEN 'Verao'
          ELSE 'Periodico'
        END as nm_disp_sazonal,
        "Unique Entry ID" as id_item,
        dt_ingestao
      FROM bronze.vestimentas
      UNION ALL
      SELECT
        "Name" as nm_item,
        "Variation" as nm_variacao,
        CASE "DIY"
          WHEN 'YES'
            THEN 'S'
          WHEN 'NO'
            THEN 'N'
          ELSE 'N'
        END as sn_reprod,
        CASE WHEN "Buy" = 'NFS'
          THEN 0
          ELSE CAST("Buy" as DECIMAL(10,2))
        END AS vl_compra,
        CASE
          WHEN "Color 1" = "Color 2"
            THEN "Color 1"
          ELSE "Color 1" || ', ' || "Color 2"
        END AS nm_cores_disponiveis,
        "Size" as tamanho,
        "Source" as nm_origem,
        CASE "Seasonal Availability"
          WHEN 'All Year' THEN 'Constante'
          WHEN 'Winter' THEN 'Inverno'
          WHEN 'Summer' THEN 'Verao'
          ELSE 'Periodico'
        END as nm_disp_sazonal,
        "Unique Entry ID" as id_item,
        dt_ingestao
      FROM bronze.bolsas
      UNION ALL
      SELECT
        "Name" as nm_item,
        "Variation" as nm_variacao,
        CASE "DIY"
          WHEN 'YES'
            THEN 'S'
          WHEN 'NO'
            THEN 'N'
          ELSE 'N'
        END as sn_reprod,
        CASE WHEN "Buy" = 'NFS'
          THEN 0
          ELSE CAST("Buy" as DECIMAL(10,2))
        END AS vl_compra,
        CASE
          WHEN "Color 1" = "Color 2"
            THEN "Color 1"
          ELSE "Color 1" || ', ' || "Color 2"
        END AS nm_cores_disponiveis,
        "Size" as tamanho,
        "Source" as nm_origem,
        CASE "Seasonal Availability"
          WHEN 'All Year' THEN 'Constante'
          WHEN 'Winter' THEN 'Inverno'
          WHEN 'Summer' THEN 'Verao'
          ELSE 'Periodico'
        END as nm_disp_sazonal,
        "Unique Entry ID" as id_item,
        dt_ingestao
      FROM bronze.calca
      UNION ALL
      SELECT
        "Name" as nm_item,
        "Variation" as nm_variacao,
        CASE "DIY"
          WHEN 'YES'
            THEN 'S'
          WHEN 'NO'
            THEN 'N'
          ELSE 'N'
        END as sn_reprod,
        CASE WHEN "Buy" = 'NFS'
          THEN 0
          ELSE CAST("Buy" as DECIMAL(10,2))
        END AS vl_compra,
        CASE
          WHEN "Color 1" = "Color 2"
            THEN "Color 1"
          ELSE "Color 1" || ', ' || "Color 2"
        END AS nm_cores_disponiveis,
        "Size" as tamanho,
        "Source" as nm_origem,
        CASE "Seasonal Availability"
          WHEN 'All Year' THEN 'Constante'
          WHEN 'Winter' THEN 'Inverno'
          WHEN 'Summer' THEN 'Verao'
          ELSE 'Periodico'
        END as nm_disp_sazonal,
        "Unique Entry ID" as id_item,
        dt_ingestao
      FROM bronze.acessorios
      UNION ALL
      SELECT
        "Name" as nm_item,
        "Variation" as nm_variacao,
        CASE "DIY"
          WHEN 'YES'
            THEN 'S'
          WHEN 'NO'
            THEN 'N'
          ELSE 'N'
        END as sn_reprod,
        CASE WHEN "Buy" = 'NFS'
          THEN 0
          ELSE CAST("Buy" as DECIMAL(10,2))
        END AS vl_compra,
        CASE
          WHEN "Color 1" = "Color 2"
            THEN "Color 1"
          ELSE "Color 1" || ', ' || "Color 2"
        END AS nm_cores_disponiveis,
        "Size" as tamanho,
        "Source" as nm_origem,
        CASE "Seasonal Availability"
          WHEN 'All Year' THEN 'Constante'
          WHEN 'Winter' THEN 'Inverno'
          WHEN 'Summer' THEN 'Verao'
          ELSE 'Periodico'
        END as nm_disp_sazonal,
        "Unique Entry ID" as id_item,
        dt_ingestao
      FROM bronze.cabeca
      UNION ALL
      SELECT
        "Name" as nm_item,
        "Variation" as nm_variacao,
        CASE "DIY"
          WHEN 'YES'
            THEN 'S'
          WHEN 'NO'
            THEN 'N'
          ELSE 'N'
        END as sn_reprod,
        CASE WHEN "Buy" = 'NFS'
          THEN 0
          ELSE CAST("Buy" as DECIMAL(10,2))
        END AS vl_compra,
        CASE
          WHEN "Color 1" = "Color 2"
            THEN "Color 1"
          ELSE "Color 1" || ', ' || "Color 2"
        END AS nm_cores_disponiveis,
        "Size" as tamanho,
        "Source" as nm_origem,
        CASE "Seasonal Availability"
          WHEN 'All Year' THEN 'Constante'
          WHEN 'Winter' THEN 'Inverno'
          WHEN 'Summer' THEN 'Verao'
          ELSE 'Periodico'
        END as nm_disp_sazonal,
        "Unique Entry ID" as id_item,
        dt_ingestao
      FROM bronze.vestimentas
    )
    SELECT
      nm_item,
      nm_variacao,
      sn_reprod,
      vl_compra,
      nm_cores_disponiveis,
      tamanho,
      nm_origem,
      nm_disp_sazonal,
      id_item,
      dt_ingestao
    FROM agrupamento
    """
)

Padronização de colunas

In [23]:
con.sql("""DESCRIBE bronze.artes""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Name               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Genuine            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Category           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Buy                │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ Sell               │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ Color 1            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Color 2            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Size               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Real Artwork Title │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

In [24]:
con.sql("""SELECT * FROM bronze.artes LIMIT 10""").show()

┌───────────────────┬─────────┬───────────────┬───────┬───────┬─────────┬─────────┬─────────┬───────────────────────────────────────────────┬─────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────┬───────────────────────────────┬───────────────┬───────────────┬────────────┬──────────┬──────────┬───────────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────────────┬───────────────────────┬────────────────────────┬───────────

In [25]:
con.execute(
    """
    CREATE TABLE IF NOT EXISTS silver.obras_arte AS
    SELECT
      "Name" as nm_item,
      CASE "Genuine"
        WHEN 'Yes' THEN 'S'
        WHEN 'No' THEN 'N'
        ELSE 'N'
      END as sn_genuina,
      CASE "Category"
        WHEN 'Wall-mounted' THEN 'Pintura'
        WHEN 'Miscellaneous' THEN 'Escultura'
        WHEN 'Housewares' THEN 'Decoracao'
        ELSE 'Outros'
      END as nm_categoria,
      CAST("Buy" as DECIMAL(10,2)) AS vl_compra,
      "Real Artwork Title" as nm_real_obra,
      "Artist" as nm_autor,
      "Unique Entry ID" as id_item,
      dt_ingestao
    FROM bronze.artes
    """
)

## Preparando nossos dados para análise - camada gold

In [26]:
con.execute("""CREATE SCHEMA IF NOT EXISTS gold""")

Catálogo de produtos de verão das Able Sisters
- somente itens de verao
- somente produtos das Able Sisters
- valor final de venda deve corresponder a pelo menos 3x o valor original
- desconto de funcionario reduz o valor final em 30%

In [ ]:
con.execute(
    """
    """
)

Curadoria de obras de arte
- somente pinturas
- somente obras genuinas
- valor de venda de 250% do valor de compra

In [29]:
con.execute(
    """
    CREATE TABLE IF NOT EXISTS gold.catalogo_pinturas_legitimas AS
    SELECT
      nm_item || ': ' || nm_real_obra as nm_peca,
      (vl_compra * 2.5) as vl_venda,
      nm_autor,
      id_item,
      dt_ingestao as dt_compra,
      CAST(NULL AS DATETIME) as dt_venda,
      NULL id_comprador
    FROM silver.obras_arte
    WHERE nm_categoria = 'Pintura'
    AND sn_genuina = 'S'
    """
)

In [30]:
con.sql("""SELECT * FROM gold.catalogo_pinturas_legitimas LIMIT 10""").show()

┌──────────────────────────────────────────────────────────────────────────────┬───────────────┬─────────────────────────────────────────────────────┬────────────────────┬────────────────────────────┬───────────┬──────────────┐
│                                   nm_peca                                    │   vl_venda    │                      nm_autor                       │      id_item       │         dt_compra          │ dt_venda  │ id_comprador │
│                                   varchar                                    │ decimal(12,3) │                       varchar                       │      varchar       │         timestamp          │ timestamp │    int32     │
├──────────────────────────────────────────────────────────────────────────────┼───────────────┼─────────────────────────────────────────────────────┼────────────────────┼────────────────────────────┼───────────┼──────────────┤
│ academic painting: Vitruvian Man                                             │     124

## Recap do que fizemos

Schemas que criamos = camadas do nosso lakehouse

In [31]:
con.sql("SELECT schema_name FROM duckdb_schemas()").show()

┌────────────────────┐
│    schema_name     │
│      varchar       │
├────────────────────┤
│ bronze             │
│ gold               │
│ main               │
│ silver             │
│ information_schema │
│ main               │
│ pg_catalog         │
│ main               │
└────────────────────┘



Tabelas que criamos

In [32]:
con.sql("SELECT schema_name, table_name FROM duckdb_tables() WHERE schema_name in ('bronze', 'silver', 'gold')").show()

┌─────────────┬─────────────────────────────┐
│ schema_name │         table_name          │
│   varchar   │           varchar           │
├─────────────┼─────────────────────────────┤
│ bronze      │ acessorios                  │
│ bronze      │ artes                       │
│ bronze      │ bolsas                      │
│ bronze      │ cabeca                      │
│ bronze      │ calca                       │
│ bronze      │ conquistas                  │
│ bronze      │ ferramentas                 │
│ bronze      │ musica                      │
│ bronze      │ personagens                 │
│ bronze      │ receitas                    │
│ bronze      │ vestimentas                 │
│ gold        │ catalogo_pinturas_legitimas │
│ silver      │ obras_arte                  │
│ silver      │ vestuario                   │
├─────────────┴─────────────────────────────┤
│ 14 rows                         2 columns │
└───────────────────────────────────────────┘

